
# 情绪与现实的博弈：GDELT 科技裁员新闻情绪数据构建

本 notebook 使用 **GDELT DOC 2.0 API** 按周抓取 2022-01-01 至 2026-04-16 的科技裁员、招聘与行业压力相关新闻元数据，只保留标题、摘要/片段、日期、来源和 URL，不抓取新闻正文。

输出文件：

- `raw_gdelt_news.csv`：原始抓取结果，包含 `query_keyword` 和 GDELT 返回字段。
- `rebuilt_news_sentiment.csv`：清洗、去重、VADER 情绪打分后的新闻级数据。
- `weekly_news_sentiment.csv`：按 Monday week 聚合后的周度情绪特征。
- `gdelt_news_data_quality_report.txt`：数据质量报告。

说明：GDELT DOC 2.0 的 `ArtList` 模式通常返回标题、URL、来源、语言、发布日期等字段，不一定返回摘要字段。如果 API 没有提供 `description/snippet`，本流程会把 `description` 留空，并用 `title + description` 计算 VADER 情绪。


In [5]:

# ================================
# 0. 依赖检查与自动安装
# ================================
# 如果当前环境缺少依赖，自动用当前 Python 解释器安装。
# pandas 如果因为 numpy 版本不匹配而无法导入，也会触发安装修复。

import importlib
import subprocess
import sys

REQUIRED_PACKAGES = [
    "numpy>=1.26,<2.0",
    "pandas",
    "requests",
    "vaderSentiment",
    "tqdm",
]

IMPORT_CHECKS = ["numpy", "pandas", "requests", "vaderSentiment", "tqdm"]

need_install = False
for module_name in IMPORT_CHECKS:
    try:
        importlib.import_module(module_name)
    except Exception as exc:
        print(f"依赖 {module_name} 导入失败，将尝试自动安装/修复。错误信息：{exc}")
        need_install = True
        break

if need_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", *REQUIRED_PACKAGES])
    print("依赖安装/修复完成。")
else:
    print("依赖检查通过。")


依赖检查通过。


In [6]:

# ================================
# 1. 导入库与全局参数
# ================================

from pathlib import Path
import json
import re
import time
from datetime import datetime

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ---------- 输出路径 ----------
# notebook 保存在 D 盘；CSV 和报告统一放到这个输出目录，方便管理。
OUTPUT_DIR = Path(r"D:\gdelt_tech_layoff_news_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_NEWS_PATH = OUTPUT_DIR / "raw_gdelt_news.csv"
REBUILT_NEWS_PATH = OUTPUT_DIR / "rebuilt_news_sentiment.csv"
WEEKLY_NEWS_PATH = OUTPUT_DIR / "weekly_news_sentiment.csv"
QUALITY_REPORT_PATH = OUTPUT_DIR / "gdelt_news_data_quality_report.txt"

# ---------- 时间范围 ----------
START_DATE = "2022-01-01"
END_DATE = "2026-04-16"

# ---------- GDELT API 参数 ----------
GDELT_DOC_API = "https://api.gdeltproject.org/api/v2/doc/doc"
MAX_RECORDS_PER_REQUEST = 250
SLEEP_SECONDS = 1.0               # 每次请求后暂停 1 秒，避免请求过快
REQUEST_TIMEOUT = 30              # 单次请求超时时间，单位：秒
MAX_RETRIES = 2                   # 单个请求失败后的重试次数；不想重试可设为 1
RETRY_SLEEP_SECONDS = 2.0
RATE_LIMIT_SLEEP_SECONDS = 5.0    # GDELT 若返回 429，会额外等待；当前 API 提示约 5 秒一次
SORT_MODE = "DateDesc"            # 按时间倒序，便于时序数据构建
USE_GDELT_LANGUAGE_FILTER = True  # 在 GDELT 查询中添加 sourcelang:eng
USE_SYSTEM_PROXY = True           # 如果代理不可用，可改为 False；在国内网络下通常需要可用代理
RUN_PREFLIGHT_CHECK = True        # 正式抓取前先做一次小请求，避免网络不可用时长时间等待

# 如果只想快速调试，可设为一个整数，例如 2 表示每个关键词只跑前 2 个周窗口。
# 正式生成数据请保持 None。
MAX_WEEK_WINDOWS_FOR_DEBUG = None

# ---------- 关键词 ----------
LAYOFF_KEYWORDS = [
    "tech layoffs",
    "technology layoffs",
    "startup layoffs",
    "AI layoffs",
    "mass layoffs",
    "job cuts",
    "workforce reduction",
    "hiring freeze",
    "downsizing",
    "restructuring",
]

HIRING_KEYWORDS = [
    "tech hiring",
    "AI hiring",
    "software engineer hiring",
    "job openings",
    "expanding workforce",
]

PRESSURE_KEYWORDS = [
    "tech downturn",
    "cost cutting",
    "economic uncertainty tech",
    "startup funding slowdown",
    "Silicon Valley layoffs",
]

ALL_KEYWORDS = LAYOFF_KEYWORDS + HIRING_KEYWORDS + PRESSURE_KEYWORDS

# ---------- 规则词表 ----------
LAYOFF_TERMS = [
    "layoff", "layoffs", "laid off", "job cuts", "workforce reduction",
    "hiring freeze", "downsizing", "restructuring", "mass layoffs",
]

HIRING_TERMS = [
    "hiring", "job openings", "recruiting", "new jobs", "expanding workforce",
]

print(f"输出目录：{OUTPUT_DIR}")
print(f"关键词数量：{len(ALL_KEYWORDS)}")
print(f"时间范围：{START_DATE} 到 {END_DATE}")


输出目录：D:\gdelt_tech_layoff_news_output
关键词数量：20
时间范围：2022-01-01 到 2026-04-16


In [7]:

# ================================
# 2. GDELT 抓取辅助函数
# ================================

def make_week_windows(start_date: str, end_date: str, days_per_window: int = 7):
    """按 7 天一组生成抓取窗口，覆盖 [start_date, end_date]。"""
    start = pd.Timestamp(start_date).normalize()
    end_exclusive = pd.Timestamp(end_date).normalize() + pd.Timedelta(days=1)

    windows = []
    current = start
    while current < end_exclusive:
        next_start = min(current + pd.Timedelta(days=days_per_window), end_exclusive)
        window_end = next_start - pd.Timedelta(seconds=1)
        windows.append((current, window_end))
        current = next_start
    return windows


def to_gdelt_datetime(ts: pd.Timestamp) -> str:
    """转换为 GDELT DOC API 需要的 YYYYMMDDHHMMSS 格式。"""
    return pd.Timestamp(ts).strftime("%Y%m%d%H%M%S")


def build_gdelt_query(keyword: str) -> str:
    """构造 GDELT 查询语句。关键词按短语匹配，并可限制英文来源。"""
    query = f'"{keyword}"'
    if USE_GDELT_LANGUAGE_FILTER:
        query = f"{query} sourcelang:eng"
    return query


def normalize_gdelt_article(article: dict, keyword: str, gdelt_query: str,
                            window_start: pd.Timestamp, window_end: pd.Timestamp,
                            request_url: str) -> dict:
    """把 GDELT 返回的单条 article 统一整理成新闻记录。"""
    # GDELT ArtList 常见字段包括 url/title/seendate/domain/language/sourcecountry 等。
    published_at = (
        article.get("seendate")
        or article.get("published_at")
        or article.get("date")
        or article.get("datetime")
    )
    description = (
        article.get("description")
        or article.get("snippet")
        or article.get("summary")
        or ""
    )
    source = (
        article.get("sourceCommonName")
        or article.get("source")
        or article.get("domain")
        or ""
    )

    row = {
        "date": published_at,
        "published_at": published_at,
        "source": source,
        "title": article.get("title", ""),
        "description": description,
        "url": article.get("url", ""),
        "query_keyword": keyword,
        "gdelt_query": gdelt_query,
        "batch_start": window_start.strftime("%Y-%m-%d"),
        "batch_end": window_end.strftime("%Y-%m-%d"),
        "request_url": request_url,
    }

    # 保留 GDELT 原始返回字段，避免后续想追溯时信息丢失。
    for key, value in article.items():
        col = f"gdelt_{key}"
        if isinstance(value, (dict, list)):
            row[col] = json.dumps(value, ensure_ascii=False)
        else:
            row[col] = value
    return row


def create_requests_session() -> requests.Session:
    """创建 requests session。可根据网络环境决定是否使用系统代理。"""
    session = requests.Session()
    session.trust_env = USE_SYSTEM_PROXY
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (compatible; academic-tech-layoff-sentiment/1.0)"
    })
    return session


def fetch_gdelt_articles(session: requests.Session, keyword: str,
                         window_start: pd.Timestamp, window_end: pd.Timestamp):
    """抓取某个关键词在某个时间窗口的 GDELT 文章列表。单个请求失败不会中断整体流程。"""
    gdelt_query = build_gdelt_query(keyword)
    params = {
        "query": gdelt_query,
        "mode": "ArtList",
        "format": "json",
        "maxrecords": MAX_RECORDS_PER_REQUEST,
        "sort": SORT_MODE,
        "startdatetime": to_gdelt_datetime(window_start),
        "enddatetime": to_gdelt_datetime(window_end),
    }

    last_error = None
    request_url = ""
    status_code = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(GDELT_DOC_API, params=params, timeout=REQUEST_TIMEOUT)
            request_url = response.url
            status_code = response.status_code
            if status_code == 429:
                last_error = f"HTTP 429 rate limited: {response.text[:200]}"
                if attempt < MAX_RETRIES:
                    time.sleep(RATE_LIMIT_SLEEP_SECONDS)
                    continue
                response.raise_for_status()
            response.raise_for_status()
            payload = response.json()
            articles = payload.get("articles", [])
            if not isinstance(articles, list):
                articles = []
            return articles, {
                "query_keyword": keyword,
                "gdelt_query": gdelt_query,
                "batch_start": window_start.strftime("%Y-%m-%d"),
                "batch_end": window_end.strftime("%Y-%m-%d"),
                "status_code": status_code,
                "n_articles": len(articles),
                "error": "",
                "request_url": request_url,
            }
        except Exception as exc:
            last_error = repr(exc)
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP_SECONDS * attempt)

    return [], {
        "query_keyword": keyword,
        "gdelt_query": gdelt_query,
        "batch_start": window_start.strftime("%Y-%m-%d"),
        "batch_end": window_end.strftime("%Y-%m-%d"),
        "status_code": status_code,
        "n_articles": 0,
        "error": last_error,
        "request_url": request_url,
    }


def gdelt_preflight_check(session: requests.Session):
    """正式抓取前做一次轻量请求，确认 GDELT API 当前可访问。"""
    test_start = pd.Timestamp(START_DATE)
    test_end = test_start + pd.Timedelta(days=6, hours=23, minutes=59, seconds=59)
    try:
        articles, log = fetch_gdelt_articles(session, ALL_KEYWORDS[0], test_start, test_end)
        if log.get("error"):
            return False, log.get("error")
        return True, f"预检查通过：HTTP {log.get('status_code')}，测试返回 {len(articles)} 条。"
    except Exception as exc:
        return False, repr(exc)


def fetch_all_gdelt_news() -> tuple[pd.DataFrame, pd.DataFrame]:
    """按周、按关键词抓取全部新闻。"""
    session = create_requests_session()

    if RUN_PREFLIGHT_CHECK:
        ok, message = gdelt_preflight_check(session)
        print(message)
        if not ok:
            raise RuntimeError(
                "GDELT 预检查失败，已停止正式抓取。请检查网络/代理后重试。"
                f"\n错误信息：{message}"
            )

    week_windows = make_week_windows(START_DATE, END_DATE)
    if MAX_WEEK_WINDOWS_FOR_DEBUG is not None:
        week_windows = week_windows[:MAX_WEEK_WINDOWS_FOR_DEBUG]

    total_requests = len(week_windows) * len(ALL_KEYWORDS)
    print(f"将执行请求数：{total_requests} = {len(week_windows)} 个周窗口 × {len(ALL_KEYWORDS)} 个关键词")

    records = []
    request_logs = []

    with tqdm(total=total_requests, desc="GDELT weekly keyword requests") as pbar:
        for keyword in ALL_KEYWORDS:
            for window_start, window_end in week_windows:
                articles, log = fetch_gdelt_articles(session, keyword, window_start, window_end)
                request_logs.append(log)

                if articles:
                    for article in articles:
                        records.append(
                            normalize_gdelt_article(
                                article=article,
                                keyword=keyword,
                                gdelt_query=log["gdelt_query"],
                                window_start=window_start,
                                window_end=window_end,
                                request_url=log.get("request_url", ""),
                            )
                        )

                # 每次请求之间暂停，避免请求过快。
                time.sleep(SLEEP_SECONDS)
                pbar.update(1)

    raw_df = pd.DataFrame(records)
    request_log_df = pd.DataFrame(request_logs)

    # 即使没有抓到记录，也保留预期列，便于后续代码可运行。
    base_columns = [
        "date", "published_at", "source", "title", "description", "url",
        "query_keyword", "gdelt_query", "batch_start", "batch_end", "request_url",
    ]
    for col in base_columns:
        if col not in raw_df.columns:
            raw_df[col] = pd.Series(dtype="object")

    return raw_df, request_log_df


In [8]:

# ================================
# 3. 执行 GDELT 抓取并保存 raw_gdelt_news.csv
# ================================
# 注意：完整参数下约为 20 个关键词 × 224 个周窗口，请求之间 sleep 1 秒，通常需要 1 小时以上。

raw_gdelt_news, gdelt_request_log = fetch_all_gdelt_news()
raw_gdelt_news.to_csv(RAW_NEWS_PATH, index=False, encoding="utf-8-sig")

print(f"raw_gdelt_news.csv 已保存：{RAW_NEWS_PATH}")
print(f"raw_gdelt_news.csv shape: {raw_gdelt_news.shape}")

# 查看失败请求数量，失败不会中断整体流程，但应在报告中关注。
failed_requests = gdelt_request_log[gdelt_request_log["error"].fillna("").ne("")]
print(f"请求总数：{len(gdelt_request_log)}")
print(f"失败请求数：{len(failed_requests)}")
if len(failed_requests) > 0:
    display(failed_requests.head(10))


预检查通过：HTTP 200，测试返回 0 条。
将执行请求数：4480 = 224 个周窗口 × 20 个关键词


GDELT weekly keyword requests: 100%|██████████| 4480/4480 [24:34:12<00:00, 19.74s/it]   


raw_gdelt_news.csv 已保存：D:\gdelt_tech_layoff_news_output\raw_gdelt_news.csv
raw_gdelt_news.csv shape: (167866, 19)
请求总数：4480
失败请求数：1974


,query_keyword,gdelt_query,batch_start,batch_end,status_code,n_articles,error,request_url
1,tech layoffs,"""tech layoffs"" sourcelang:eng",2022-01-08,2022-01-14,429.0,0,HTTPError('429 Client Error: Too Many Requests...,https://api.gdeltproject.org/api/v2/doc/doc?qu...
2,tech layoffs,"""tech layoffs"" sourcelang:eng",2022-01-15,2022-01-21,429.0,0,HTTPError('429 Client Error: Too Many Requests...,https://api.gdeltproject.org/api/v2/doc/doc?qu...
7,tech layoffs,"""tech layoffs"" sourcelang:eng",2022-02-19,2022-02-25,429.0,0,HTTPError('429 Client Error: Too Many Requests...,https://api.gdeltproject.org/api/v2/doc/doc?qu...
8,tech layoffs,"""tech layoffs"" sourcelang:eng",2022-02-26,2022-03-04,429.0,0,HTTPError('429 Client Error: Too Many Requests...,https://api.gdeltproject.org/api/v2/doc/doc?qu...
9,tech layoffs,"""tech layoffs"" sourcelang:eng",2022-03-05,2022-03-11,429.0,0,HTTPError('429 Client Error: Too Many Requests...,https://api.gdeltproject.org/api/v2/doc/doc?qu...
11,tech layoffs,"""tech layoffs"" sourcelang:eng",2022-03-19,2022-03-25,429.0,0,HTTPError('429 Client Error: Too Many Requests...,https://api.gdeltproject.org/api/v2/doc/doc?qu...
12,tech layoffs,"""tech layoffs"" sourcelang:eng",2022-03-26,2022-04-01,429.0,0,HTTPError('429 Client Error: Too Many Requests...,https://api.gdeltproject.org/api/v2/doc/doc?qu...
14,tech layoffs,"""tech layoffs"" sourcelang:eng",2022-04-09,2022-04-15,429.0,0,HTTPError('429 Client Error: Too Many Requests...,https://api.gdeltproject.org/api/v2/doc/doc?qu...
15,tech layoffs,"""tech layoffs"" sourcelang:eng",2022-04-16,2022-04-22,429.0,0,HTTPError('429 Client Error: Too Many Requests...,https://api.gdeltproject.org/api/v2/doc/doc?qu...
16,tech layoffs,"""tech layoffs"" sourcelang:eng",2022-04-23,2022-04-29,429.0,0,HTTPError('429 Client Error: Too Many Requests...,https://api.gdeltproject.org/api/v2/doc/doc?qu...


In [9]:

# ================================
# 4. 清洗、去重、VADER 情绪打分
# ================================

def parse_published_at(series: pd.Series) -> pd.Series:
    """兼容 GDELT seendate（如 20240131T143000Z）和常见日期字符串。"""
    text = series.astype("string").str.strip()
    parsed = pd.to_datetime(text, format="%Y%m%dT%H%M%SZ", errors="coerce", utc=True)
    fallback = pd.to_datetime(text.where(parsed.isna()), errors="coerce", utc=True)
    return parsed.fillna(fallback)


def build_regex_from_terms(terms: list[str]) -> str:
    """把规则词表转换为不区分大小写的正则表达式。"""
    escaped_terms = [re.escape(term) for term in terms]
    return r"(" + r"|".join(escaped_terms) + r")"


def classify_sentiment(score: float) -> str:
    """按照 VADER compound score 分类。"""
    if pd.isna(score):
        return np.nan
    if score <= -0.05:
        return "negative"
    if score >= 0.05:
        return "positive"
    return "neutral"


def clean_and_score_news(raw_df: pd.DataFrame) -> pd.DataFrame:
    """清洗原始新闻并计算情绪分数。"""
    output_columns = [
        "date", "published_at", "source", "title", "description", "url", "query_keyword",
        "sentiment", "sentiment_cat", "is_layoff_news", "is_hiring_news",
    ]

    if raw_df.empty:
        return pd.DataFrame(columns=output_columns)

    df = raw_df.copy()

    # 确保核心列存在。
    for col in ["date", "published_at", "source", "title", "description", "url", "query_keyword"]:
        if col not in df.columns:
            df[col] = ""

    # 只保留有 title 和 url 的记录。
    for col in ["title", "description", "url", "source", "query_keyword"]:
        df[col] = df[col].fillna("").astype(str).str.strip()
    df = df[df["title"].ne("")]
    df = df[df["url"].ne("")]

    # 如果 GDELT 返回 language 字段，则保留 English / en；空值保留，避免字段缺失导致误删。
    language_col = None
    for candidate in ["language", "gdelt_language"]:
        if candidate in df.columns:
            language_col = candidate
            break
    if language_col is not None and df[language_col].notna().any():
        lang = df[language_col].fillna("").astype(str).str.lower().str.strip()
        df = df[lang.eq("") | lang.isin(["english", "en", "eng"])]

    # 日期解析与时间范围过滤。
    published_dt = parse_published_at(df["published_at"])
    df = df[published_dt.notna()].copy()
    published_dt = published_dt.loc[df.index]
    df["published_at_dt"] = published_dt
    df["date"] = df["published_at_dt"].dt.tz_convert(None).dt.normalize()
    df["published_at"] = df["published_at_dt"].dt.strftime("%Y-%m-%d %H:%M:%S UTC")

    start_ts = pd.Timestamp(START_DATE)
    end_ts = pd.Timestamp(END_DATE)
    df = df[(df["date"] >= start_ts) & (df["date"] <= end_ts)].copy()

    # 按 URL 去重。如果同一新闻被多个关键词抓到，只保留第一条。
    df["url"] = df["url"].str.strip()
    df = df.sort_values(["published_at_dt", "query_keyword", "url"], ascending=[True, True, True])
    df = df.drop_duplicates(subset=["url"], keep="first").copy()

    # 构造文本字段：GDELT 若无 description，则主要使用 title。
    text_for_nlp = (df["title"].fillna("") + ". " + df["description"].fillna("")).str.strip()

    analyzer = SentimentIntensityAnalyzer()
    df["sentiment"] = text_for_nlp.apply(lambda text: analyzer.polarity_scores(text)["compound"])
    df["sentiment_cat"] = df["sentiment"].apply(classify_sentiment)

    layoff_pattern = build_regex_from_terms(LAYOFF_TERMS)
    hiring_pattern = build_regex_from_terms(HIRING_TERMS)
    lower_text = text_for_nlp.str.lower()

    df["is_layoff_news"] = lower_text.str.contains(layoff_pattern, case=False, regex=True, na=False)
    df["is_hiring_news"] = lower_text.str.contains(hiring_pattern, case=False, regex=True, na=False)

    rebuilt = df[output_columns].copy()
    return rebuilt


rebuilt_news_sentiment = clean_and_score_news(raw_gdelt_news)
rebuilt_news_sentiment.to_csv(REBUILT_NEWS_PATH, index=False, encoding="utf-8-sig")

print(f"rebuilt_news_sentiment.csv 已保存：{REBUILT_NEWS_PATH}")
print(f"rebuilt_news_sentiment.csv shape: {rebuilt_news_sentiment.shape}")
rebuilt_news_sentiment.head()


C:\Users\NJL\AppData\Local\Temp\ipykernel_80148\2895047862.py:91: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["is_layoff_news"] = lower_text.str.contains(layoff_pattern, case=False, regex=True, na=False)
C:\Users\NJL\AppData\Local\Temp\ipykernel_80148\2895047862.py:92: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["is_hiring_news"] = lower_text.str.contains(hiring_pattern, case=False, regex=True, na=False)


rebuilt_news_sentiment.csv 已保存：D:\gdelt_tech_layoff_news_output\rebuilt_news_sentiment.csv
rebuilt_news_sentiment.csv shape: (160711, 11)


,date,published_at,source,title,description,url,query_keyword,sentiment,sentiment_cat,is_layoff_news,is_hiring_news
20018,2022-01-01,2022-01-01 00:00:00 UTC,bruneinews.net,News outlets hurt by dwindling public interest...,,https://www.bruneinews.net/news/272074475/outl...,job cuts,-0.1027,negative,False,False
20017,2022-01-01,2022-01-01 00:00:00 UTC,laosnews.net,News outlets hurt by dwindling public interest...,,https://www.laosnews.net/news/272074475/outlet...,job cuts,-0.1027,negative,False,False
67171,2022-01-01,2022-01-01 00:30:00 UTC,journalismpakistan.com,Dunya News presenter Kamran Shahid slapped wit...,,http://www.journalismpakistan.com/dunya-news-p...,downsizing,-0.3612,negative,False,False
67170,2022-01-01,2022-01-01 03:00:00 UTC,wscountytimes.co.uk,Sussex PR company founder and friend of Dame V...,,https://www.wscountytimes.co.uk/news/people/su...,downsizing,0.7506,positive,False,False
67169,2022-01-01,2022-01-01 03:30:00 UTC,eurasiantimes.com,F - 15 Eagles Set For A Big Boost ; Boeing To ...,,https://eurasiantimes.com/japans-f-15-eagles-s...,downsizing,0.7650,positive,False,False


In [10]:

# ================================
# 5. 构造 Monday week 的周度新闻情绪特征
# ================================

def monday_week_start(date_series: pd.Series) -> pd.Series:
    """把日期转换为所在周的 Monday。"""
    date_series = pd.to_datetime(date_series, errors="coerce")
    return date_series - pd.to_timedelta(date_series.dt.weekday, unit="D")


def make_all_monday_weeks(start_date: str, end_date: str) -> pd.DataFrame:
    """生成完整 Monday week 序列；无新闻的周也保留，情绪均值保持 NaN。"""
    start_ts = pd.Timestamp(start_date).normalize()
    end_ts = pd.Timestamp(end_date).normalize()
    start_monday = start_ts - pd.Timedelta(days=start_ts.weekday())
    end_monday = end_ts - pd.Timedelta(days=end_ts.weekday())
    return pd.DataFrame({"week": pd.date_range(start_monday, end_monday, freq="W-MON")})


def build_weekly_news_features(news_df: pd.DataFrame) -> pd.DataFrame:
    """按周聚合新闻数量、情绪和裁员/招聘新闻占比。"""
    all_weeks = make_all_monday_weeks(START_DATE, END_DATE)

    count_cols = [
        "weekly_news_count",
        "weekly_layoff_news_count",
        "weekly_hiring_news_count",
        "weekly_negative_news_count",
        "weekly_positive_news_count",
        "weekly_neutral_news_count",
    ]

    mean_cols = [
        "weekly_avg_sentiment",
        "weekly_layoff_avg_sentiment",
        "weekly_hiring_avg_sentiment",
    ]

    if news_df.empty:
        weekly = all_weeks.copy()
        for col in count_cols:
            weekly[col] = 0
        for col in mean_cols + ["weekly_negative_share", "weekly_layoff_news_share", "sentiment_shock"]:
            weekly[col] = np.nan
        return weekly

    df = news_df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df[df["date"].notna()].copy()
    df["week"] = monday_week_start(df["date"])

    base = df.groupby("week", as_index=False).agg(
        weekly_news_count=("url", "count"),
        weekly_layoff_news_count=("is_layoff_news", "sum"),
        weekly_hiring_news_count=("is_hiring_news", "sum"),
        weekly_avg_sentiment=("sentiment", "mean"),
    )

    sentiment_counts = (
        df.groupby(["week", "sentiment_cat"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    for cat in ["negative", "positive", "neutral"]:
        if cat not in sentiment_counts.columns:
            sentiment_counts[cat] = 0
    sentiment_counts = sentiment_counts.rename(columns={
        "negative": "weekly_negative_news_count",
        "positive": "weekly_positive_news_count",
        "neutral": "weekly_neutral_news_count",
    })[[
        "week",
        "weekly_negative_news_count",
        "weekly_positive_news_count",
        "weekly_neutral_news_count",
    ]]

    layoff_avg = (
        df[df["is_layoff_news"]]
        .groupby("week")["sentiment"]
        .mean()
        .rename("weekly_layoff_avg_sentiment")
        .reset_index()
    )
    hiring_avg = (
        df[df["is_hiring_news"]]
        .groupby("week")["sentiment"]
        .mean()
        .rename("weekly_hiring_avg_sentiment")
        .reset_index()
    )

    weekly = (
        all_weeks
        .merge(base, on="week", how="left")
        .merge(sentiment_counts, on="week", how="left")
        .merge(layoff_avg, on="week", how="left")
        .merge(hiring_avg, on="week", how="left")
    )

    for col in count_cols:
        weekly[col] = weekly[col].fillna(0).astype(int)

    weekly["weekly_negative_share"] = np.where(
        weekly["weekly_news_count"] > 0,
        weekly["weekly_negative_news_count"] / weekly["weekly_news_count"],
        np.nan,
    )
    weekly["weekly_layoff_news_share"] = np.where(
        weekly["weekly_news_count"] > 0,
        weekly["weekly_layoff_news_count"] / weekly["weekly_news_count"],
        np.nan,
    )

    # 情绪冲击：本周平均情绪 - 上一周平均情绪。无新闻周保持 NaN，不强行当作中性。
    weekly["sentiment_shock"] = weekly["weekly_avg_sentiment"].diff()

    ordered_cols = [
        "week",
        "weekly_news_count",
        "weekly_layoff_news_count",
        "weekly_hiring_news_count",
        "weekly_negative_news_count",
        "weekly_positive_news_count",
        "weekly_neutral_news_count",
        "weekly_avg_sentiment",
        "weekly_layoff_avg_sentiment",
        "weekly_hiring_avg_sentiment",
        "weekly_negative_share",
        "weekly_layoff_news_share",
        "sentiment_shock",
    ]
    return weekly[ordered_cols]


weekly_news_sentiment = build_weekly_news_features(rebuilt_news_sentiment)
weekly_news_sentiment.to_csv(WEEKLY_NEWS_PATH, index=False, encoding="utf-8-sig")

print(f"weekly_news_sentiment.csv 已保存：{WEEKLY_NEWS_PATH}")
print(f"weekly_news_sentiment.csv shape: {weekly_news_sentiment.shape}")
weekly_news_sentiment.head()


weekly_news_sentiment.csv 已保存：D:\gdelt_tech_layoff_news_output\weekly_news_sentiment.csv
weekly_news_sentiment.csv shape: (225, 13)


,week,weekly_news_count,weekly_layoff_news_count,weekly_hiring_news_count,weekly_negative_news_count,weekly_positive_news_count,weekly_neutral_news_count,weekly_avg_sentiment,weekly_layoff_avg_sentiment,weekly_hiring_avg_sentiment,weekly_negative_share,weekly_layoff_news_share,sentiment_shock
0,2021-12-27,41,1,0,8,13,20,0.095649,0.456700,NaN,0.195122,0.024390,NaN
1,2022-01-03,652,19,4,173,228,251,0.010158,-0.392184,0.375825,0.265337,0.029141,-0.085490
2,2022-01-10,956,19,11,279,235,442,0.025468,-0.059437,0.246427,0.291841,0.019874,0.015309
3,2022-01-17,975,25,54,291,266,418,-0.016301,-0.244096,-0.047728,0.298462,0.025641,-0.041769
4,2022-01-24,773,33,17,242,200,331,0.001317,-0.170585,0.319282,0.313066,0.042691,0.017618


In [11]:

# ================================
# 6. 生成数据质量报告
# ================================

def series_to_text(series: pd.Series) -> str:
    """把 value_counts / describe 等结果转成文本。"""
    if series is None or len(series) == 0:
        return "（空）"
    return series.to_string()


def make_data_quality_report(raw_df: pd.DataFrame, rebuilt_df: pd.DataFrame,
                             weekly_df: pd.DataFrame, request_log_df: pd.DataFrame) -> str:
    """生成中文数据质量报告。"""
    report_lines = []
    report_lines.append("GDELT 科技裁员新闻情绪数据质量报告")
    report_lines.append("=" * 60)
    report_lines.append(f"生成时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report_lines.append(f"GDELT 时间范围：{START_DATE} 到 {END_DATE}")
    report_lines.append("")

    total_raw = len(raw_df)
    total_rebuilt = len(rebuilt_df)
    report_lines.append("一、总体规模")
    report_lines.append(f"总新闻条数（raw，含重复 URL）：{total_raw}")
    report_lines.append(f"去重后新闻条数（rebuilt）：{total_rebuilt}")
    if len(request_log_df) > 0 and "error" in request_log_df.columns:
        failed_count = request_log_df[request_log_df["error"].fillna("").ne("")].shape[0]
        report_lines.append(f"请求总数：{len(request_log_df)}")
        report_lines.append(f"失败请求数：{failed_count}")
    report_lines.append("")

    report_lines.append("二、时间范围")
    if total_rebuilt > 0:
        dates = pd.to_datetime(rebuilt_df["date"], errors="coerce")
        report_lines.append(f"清洗后新闻最早日期：{dates.min().date() if dates.notna().any() else 'NA'}")
        report_lines.append(f"清洗后新闻最晚日期：{dates.max().date() if dates.notna().any() else 'NA'}")
    else:
        report_lines.append("清洗后新闻最早日期：NA")
        report_lines.append("清洗后新闻最晚日期：NA")
    report_lines.append("")

    year_index = list(range(pd.Timestamp(START_DATE).year, pd.Timestamp(END_DATE).year + 1))
    if total_rebuilt > 0:
        tmp = rebuilt_df.copy()
        tmp["year"] = pd.to_datetime(tmp["date"], errors="coerce").dt.year
        yearly_news = tmp["year"].value_counts().sort_index().reindex(year_index, fill_value=0)
        yearly_layoff_news = (
            tmp[tmp["is_layoff_news"]]
            ["year"]
            .value_counts()
            .sort_index()
            .reindex(year_index, fill_value=0)
        )
    else:
        yearly_news = pd.Series(0, index=year_index)
        yearly_layoff_news = pd.Series(0, index=year_index)

    report_lines.append("三、每年新闻数量")
    report_lines.append(series_to_text(yearly_news))
    report_lines.append("")

    report_lines.append("四、每年裁员相关新闻数量")
    report_lines.append(series_to_text(yearly_layoff_news))
    report_lines.append("")

    report_lines.append("五、每周新闻覆盖情况")
    total_weeks = len(weekly_df)
    has_news_weeks = int((weekly_df["weekly_news_count"] > 0).sum()) if total_weeks > 0 else 0
    no_news_weeks = total_weeks - has_news_weeks
    coverage_rate = has_news_weeks / total_weeks if total_weeks > 0 else np.nan
    report_lines.append(f"总周数：{total_weeks}")
    report_lines.append(f"有新闻记录的周数：{has_news_weeks}")
    report_lines.append(f"没有新闻记录的周数：{no_news_weeks}")
    report_lines.append(f"周覆盖率：{coverage_rate:.2%}" if pd.notna(coverage_rate) else "周覆盖率：NA")
    weekly_coverage = weekly_df[[
        "week", "weekly_news_count", "weekly_layoff_news_count", "weekly_hiring_news_count"
    ]].copy()
    weekly_coverage["week"] = pd.to_datetime(weekly_coverage["week"]).dt.strftime("%Y-%m-%d")
    report_lines.append("\n每周覆盖明细：")
    report_lines.append(weekly_coverage.to_string(index=False))
    report_lines.append("")

    report_lines.append("六、每个关键词抓到的新闻数量（raw，含重复 URL）")
    if total_raw > 0 and "query_keyword" in raw_df.columns:
        keyword_counts = raw_df["query_keyword"].value_counts().reindex(ALL_KEYWORDS, fill_value=0)
    else:
        keyword_counts = pd.Series(0, index=ALL_KEYWORDS)
    report_lines.append(series_to_text(keyword_counts))
    report_lines.append("")

    report_lines.append("七、Top 20 新闻来源（rebuilt，去重后）")
    if total_rebuilt > 0:
        top_sources = rebuilt_df["source"].replace("", np.nan).dropna().value_counts().head(20)
    else:
        top_sources = pd.Series(dtype="int64")
    report_lines.append(series_to_text(top_sources))
    report_lines.append("")

    report_lines.append("八、sentiment 描述性统计")
    if total_rebuilt > 0:
        report_lines.append(rebuilt_df["sentiment"].describe().to_string())
    else:
        report_lines.append("（空）")
    report_lines.append("")

    report_lines.append("九、sentiment_cat 频数分布")
    if total_rebuilt > 0:
        sentiment_cat_counts = rebuilt_df["sentiment_cat"].value_counts().reindex(
            ["negative", "neutral", "positive"], fill_value=0
        )
    else:
        sentiment_cat_counts = pd.Series(0, index=["negative", "neutral", "positive"])
    report_lines.append(series_to_text(sentiment_cat_counts))
    report_lines.append("")

    return "\n".join(report_lines)


quality_report_text = make_data_quality_report(
    raw_df=raw_gdelt_news,
    rebuilt_df=rebuilt_news_sentiment,
    weekly_df=weekly_news_sentiment,
    request_log_df=gdelt_request_log,
)

QUALITY_REPORT_PATH.write_text(quality_report_text, encoding="utf-8-sig")
print(f"数据质量报告已保存：{QUALITY_REPORT_PATH}")
print("报告前 1000 字预览：")
print(quality_report_text[:1000])


数据质量报告已保存：D:\gdelt_tech_layoff_news_output\gdelt_news_data_quality_report.txt
报告前 1000 字预览：
GDELT 科技裁员新闻情绪数据质量报告
生成时间：2026-05-13 17:50:20
GDELT 时间范围：2022-01-01 到 2026-04-16

一、总体规模
总新闻条数（raw，含重复 URL）：167866
去重后新闻条数（rebuilt）：160711
请求总数：4480
失败请求数：1974

二、时间范围
清洗后新闻最早日期：2022-01-01
清洗后新闻最晚日期：2026-04-16

三、每年新闻数量
year
2022    40375
2023    34474
2024    33013
2025    43190
2026     9659

四、每年裁员相关新闻数量
year
2022    2833
2023    3871
2024    3168
2025    3990
2026     979

五、每周新闻覆盖情况
总周数：225
有新闻记录的周数：223
没有新闻记录的周数：2
周覆盖率：99.11%

每周覆盖明细：
      week  weekly_news_count  weekly_layoff_news_count  weekly_hiring_news_count
2021-12-27                 41                         1                         0
2022-01-03                652                        19                         4
2022-01-10                956                        19                        11
2022-01-17                975                        25                        54
2022-01-24                773                        

In [12]:

# ================================
# 7. 按要求打印最终结果
# ================================

print(f"raw_gdelt_news.csv shape: {raw_gdelt_news.shape}")
print(f"rebuilt_news_sentiment.csv shape: {rebuilt_news_sentiment.shape}")
print(f"weekly_news_sentiment.csv shape: {weekly_news_sentiment.shape}")
print("\nweekly_news_sentiment.csv 前 5 行：")
display(weekly_news_sentiment.head(5))


raw_gdelt_news.csv shape: (167866, 19)
rebuilt_news_sentiment.csv shape: (160711, 11)
weekly_news_sentiment.csv shape: (225, 13)

weekly_news_sentiment.csv 前 5 行：


,week,weekly_news_count,weekly_layoff_news_count,weekly_hiring_news_count,weekly_negative_news_count,weekly_positive_news_count,weekly_neutral_news_count,weekly_avg_sentiment,weekly_layoff_avg_sentiment,weekly_hiring_avg_sentiment,weekly_negative_share,weekly_layoff_news_share,sentiment_shock
0,2021-12-27,41,1,0,8,13,20,0.095649,0.456700,NaN,0.195122,0.024390,NaN
1,2022-01-03,652,19,4,173,228,251,0.010158,-0.392184,0.375825,0.265337,0.029141,-0.085490
2,2022-01-10,956,19,11,279,235,442,0.025468,-0.059437,0.246427,0.291841,0.019874,0.015309
3,2022-01-17,975,25,54,291,266,418,-0.016301,-0.244096,-0.047728,0.298462,0.025641,-0.041769
4,2022-01-24,773,33,17,242,200,331,0.001317,-0.170585,0.319282,0.313066,0.042691,0.017618



## 与 `layoffs_events.csv` 的合并逻辑

下面这格代码展示如何把 `weekly_news_sentiment.csv` 合并到 `layoffs_events.csv` 的周度裁员事件数据中。

关键原则：

- `layoffs_events.csv` 的 `date` 转为 datetime。
- 用 Monday 作为周开始日期，聚合出 `weekly_layoff_event_count` 和 `weekly_known_layoff_count`。
- 以 `weekly_layoffs` 为主表，按 `week` 左连接 `weekly_news_sentiment.csv`。
- 没有新闻的周不要当作中性情绪。
- 新增 `has_news_week = 1/0`。
- 新闻数量类变量可以填 0。
- 情绪均值、占比和 `sentiment_shock` 保留 NaN，建模时再谨慎处理。


In [14]:

# ================================
# 8. 与 layoffs_events.csv 合并示例
# ================================
# 运行前请把 LAYOFFS_EVENTS_PATH 改成你的 layoffs_events.csv 实际路径。
# 如果文件就在当前工作目录或输出目录，下面的自动搜索会尝试找到它。

possible_layoffs_paths = [
    OUTPUT_DIR / "layoffs_events.csv",
    Path.cwd() / "layoffs_events.csv",
    Path(r"D:\VS_codex\layoffs_events.csv"),
    Path(r"D:\机器学习\data\layoffs_events - 副本.csv"),
]
LAYOFFS_EVENTS_PATH = next((p for p in possible_layoffs_paths if p.exists()), possible_layoffs_paths[0])

if not LAYOFFS_EVENTS_PATH.exists():
    print("未找到 layoffs_events.csv。请把 LAYOFFS_EVENTS_PATH 改成实际路径后重新运行本单元。")
    print(f"当前设置：{LAYOFFS_EVENTS_PATH}")
else:
    layoffs_events = pd.read_csv(LAYOFFS_EVENTS_PATH)
    layoffs_events["date"] = pd.to_datetime(layoffs_events["date"], errors="coerce")
    layoffs_events = layoffs_events[layoffs_events["date"].notna()].copy()
    layoffs_events["week"] = monday_week_start(layoffs_events["date"])

    # 自动识别裁员人数列。常见 Kaggle layoff.fyi 数据列名是 total_laid_off。
    layoff_count_candidates = [
        "total_laid_off", "laid_off", "num_laid_off", "layoff_count",
        "known_layoff_count", "employees_laid_off",
    ]
    layoff_count_col = next((c for c in layoff_count_candidates if c in layoffs_events.columns), None)

    if layoff_count_col is not None:
        layoffs_events[layoff_count_col] = pd.to_numeric(layoffs_events[layoff_count_col], errors="coerce")
        weekly_known = layoffs_events.groupby("week", as_index=False)[layoff_count_col].sum(min_count=1)
        weekly_known = weekly_known.rename(columns={layoff_count_col: "weekly_known_layoff_count"})
    else:
        weekly_known = layoffs_events[["week"]].drop_duplicates().copy()
        weekly_known["weekly_known_layoff_count"] = np.nan
        print("没有识别到裁员人数列，因此 weekly_known_layoff_count 保留为 NaN。")

    weekly_events = (
        layoffs_events
        .groupby("week", as_index=False)
        .size()
        .rename(columns={"size": "weekly_layoff_event_count"})
    )

    weekly_layoffs = weekly_events.merge(weekly_known, on="week", how="left")

    weekly_news = pd.read_csv(WEEKLY_NEWS_PATH)
    weekly_news["week"] = pd.to_datetime(weekly_news["week"], errors="coerce")

    modeling_df = weekly_layoffs.merge(weekly_news, on="week", how="left")

    # 如果 weekly_news_sentiment.csv 包含完整周序列，则 weekly_news_count=0 代表无新闻；
    # 如果只包含有新闻的周，merge 后 NaN 也代表无新闻。
    modeling_df["has_news_week"] = (modeling_df["weekly_news_count"].fillna(0) > 0).astype(int)

    news_count_cols = [
        "weekly_news_count",
        "weekly_layoff_news_count",
        "weekly_hiring_news_count",
        "weekly_negative_news_count",
        "weekly_positive_news_count",
        "weekly_neutral_news_count",
    ]
    for col in news_count_cols:
        if col in modeling_df.columns:
            modeling_df[col] = modeling_df[col].fillna(0).astype(int)

    # 注意：以下情绪均值/占比/冲击变量不要因为无新闻而填成 0 或中性。
    # weekly_avg_sentiment、weekly_layoff_avg_sentiment、weekly_hiring_avg_sentiment、
    # weekly_negative_share、weekly_layoff_news_share、sentiment_shock 保留 NaN。

    MODELING_DATA_PATH = OUTPUT_DIR / "weekly_layoffs_with_news_sentiment.csv"
    modeling_df.to_csv(MODELING_DATA_PATH, index=False, encoding="utf-8-sig")

    print(f"合并后的建模数据已保存：{MODELING_DATA_PATH}")
    print(f"weekly_layoffs shape: {weekly_layoffs.shape}")
    print(f"modeling_df shape: {modeling_df.shape}")
    display(modeling_df.head())


合并后的建模数据已保存：D:\gdelt_tech_layoff_news_output\weekly_layoffs_with_news_sentiment.csv
weekly_layoffs shape: (204, 3)
modeling_df shape: (204, 16)


,week,weekly_layoff_event_count,weekly_known_layoff_count,weekly_news_count,weekly_layoff_news_count,weekly_hiring_news_count,weekly_negative_news_count,weekly_positive_news_count,weekly_neutral_news_count,weekly_avg_sentiment,weekly_layoff_avg_sentiment,weekly_hiring_avg_sentiment,weekly_negative_share,weekly_layoff_news_share,sentiment_shock,has_news_week
0,2020-03-09,2,14.0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2020-03-16,17,1315.0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2020-03-23,71,6400.0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
3,2020-03-30,29,1804.0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
4,2020-04-13,51,6107.0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,0
